# Stage 2 – Method 2 Discriminator
Latent discriminator with reconstruction + classification losses.

In [ ]:

import os
import sys
import shutil
from pathlib import Path


use_colab = "google.colab" in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    project_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    project_dir = Path.cwd()


In [ ]:

import os
import sys
import shutil
from pathlib import Path

# project_dir is set in the previous cell
lib_dir = project_dir / 'lib'
sys.path.insert(0, str(lib_dir))
sys.path.insert(0, str(project_dir))

drive_root = project_dir / 'datasets'
drive_zip = drive_root / f"{DATASET_NAME}.zip"
local_root = Path('/content/datasets') if "google.colab" in sys.modules else drive_root
if "google.colab" in sys.modules:
    local_root.mkdir(parents=True, exist_ok=True)
local_zip = local_root / f"{DATASET_NAME}.zip"
extract_dir = local_root / DATASET_NAME
nested_dir = extract_dir / DATASET_NAME

def _has_npz(p: Path) -> bool:
    return p.is_dir() and any(p.rglob('*.npz'))

# Sync zip to local if available
if drive_zip.exists() and (not local_zip.exists() or drive_zip.stat().st_mtime > local_zip.stat().st_mtime):
    shutil.copy2(drive_zip, local_zip)

# Resolve data_dir (prefer local extracted; else extract local zip; else drive extracted)
if _has_npz(extract_dir):
    data_dir = extract_dir
elif _has_npz(nested_dir):
    data_dir = nested_dir
elif local_zip.exists():
    shutil.unpack_archive(str(local_zip), str(local_root))
    if _has_npz(nested_dir):
        data_dir = nested_dir
    elif _has_npz(extract_dir):
        data_dir = extract_dir
    else:
        data_dir = None
else:
    data_dir = None

if data_dir is None:
    drive_extract = drive_root / DATASET_NAME
    drive_nested = drive_extract / DATASET_NAME
    if _has_npz(drive_nested):
        data_dir = drive_nested
    elif _has_npz(drive_extract):
        data_dir = drive_extract

if data_dir is None:
    raise FileNotFoundError(f"Expected {DATASET_NAME} zip/extracted under {drive_root} or local {local_root}")

os.environ['BANDVAE_DATA_DIR'] = str(data_dir)

split_json = project_dir / 'data_split.json'
base_data_dir = data_dir.parent if (data_dir.parent / 'test_balanced_npz').exists() else data_dir
pretrained_path = project_dir / 'runs' / 'stage1_pretrain_colab' / 'stage1_pretrained.pt'
save_dir = project_dir / 'runs' / 'stage2_method2_discriminator_colab'
save_dir.mkdir(parents=True, exist_ok=True)
os.chdir(project_dir)
print('Split JSON:', split_json)
print('Pretrained:', pretrained_path)
print('Save dir:', save_dir)
print(f'Project dir: {project_dir}')
print(f'Data dir: {data_dir}')


In [ ]:

import time
import torch

import torch.optim as optim
from torch.utils.data import DataLoader

from config_bandvae import get_config
from dataset_stage2 import Stage2Dataset
from model_bandvae import BandSplitVAE, band_split_vae_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = get_config('full')
config.T_fixed = 300
config.fc_low = 2.0
config.fc_high = 8.0
config.filter_order = 4
config.C_h = 48
config.C_z = 12
config.dilations = [1, 2, 4]
config.lr = 1e-4
config.epochs = 10
config.batch_size = 64
config.num_workers = 4
config.device = device
print(config)


In [ ]:

train_dataset = Stage2Dataset(
    split_json_path=str(split_json),
    split_name='stage2_train',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=True,
    base_dir=str(base_data_dir),
)

val_dataset = Stage2Dataset(
    split_json_path=str(split_json),
    split_name='final_test',
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=False,
    base_dir=str(base_data_dir),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print('Train samples:', len(train_dataset), 'Val samples:', len(val_dataset))


In [ ]:

checkpoint = torch.load(pretrained_path, map_location=config.device)

model = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations,
).to(config.device)
model.load_state_dict(checkpoint['model_state_dict'])

optimizer = optim.Adam(model.parameters(), lr=config.lr)
use_amp = config.device == 'cuda'


In [ ]:

import torch.nn as nn

LAMBDA_CLS = 1.0

class LatentDiscriminator(nn.Module):
    def __init__(self, C_z, num_bands=3, hidden_dim=128):
        super().__init__()
        input_dim = C_z * 2 * num_bands
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, z_lf, z_bp, z_hf):
        def pool(z):
            if z.dim() == 3:
                z_avg = z.mean(dim=2)
                z_max = z.max(dim=2)[0]
                return torch.cat([z_avg, z_max], dim=1)
            return z

        z_concat = torch.cat([pool(z_lf), pool(z_bp), pool(z_hf)], dim=1)
        return self.net(z_concat)


discriminator = LatentDiscriminator(config.C_z).to(config.device)
optimizer = optim.Adam(list(model.parameters()) + list(discriminator.parameters()), lr=config.lr)
use_amp = config.device == 'cuda'
criterion_cls = nn.BCEWithLogitsLoss()


In [ ]:


@torch.no_grad()
def validate(model, discriminator, loader, device):
    model.eval()
    discriminator.eval()
    total_loss = rec_loss = cls_loss = 0.0
    n_batches = len(loader)
    correct = total = 0

    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device).float()

        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}

        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )

        logits = discriminator(mus['lf'], mus['bp'], mus['hf']).squeeze(1)
        cls = criterion_cls(logits, labels)
        total_batch = loss + LAMBDA_CLS * cls

        total_loss += total_batch.item()
        rec_loss += loss_dict['total']
        cls_loss += cls.item()
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

    acc = correct / total if total else 0.0
    return {
        'total': total_loss / n_batches,
        'rec': rec_loss / n_batches,
        'cls': cls_loss / n_batches,
        'accuracy': acc,
    }


def train_epoch(model, discriminator, loader, optimizer, device, use_amp=False):
    model.train()
    discriminator.train()
    total_loss = rec_loss = cls_loss = 0.0
    n_batches = len(loader)
    correct = total = 0

    for x_lf, x_bp, x_hf, labels in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        labels = labels.to(device).float()
        optimizer.zero_grad()

        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )
        logits = discriminator(mus['lf'], mus['bp'], mus['hf']).squeeze(1)
        cls = criterion_cls(logits, labels)
        total_batch = loss + LAMBDA_CLS * cls
        total_batch.backward()
        optimizer.step()

        total_loss += total_batch.item()
        rec_loss += loss_dict['total']
        cls_loss += cls.item()
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

    acc = correct / total if total else 0.0
    return {
        'total': total_loss / n_batches,
        'rec': rec_loss / n_batches,
        'cls': cls_loss / n_batches,
        'accuracy': acc,
    }

In [ ]:

best_acc = 0.0
best_path = save_dir / 'stage2_method2_discriminator_best.pt'

for epoch in range(1, config.epochs + 1):
    start = time.time()
    train_loss = train_epoch(model, discriminator, train_loader, optimizer, config.device, use_amp=use_amp)
    val_loss = validate(model, discriminator, val_loader, config.device)
    duration = (time.time() - start) / 60

    print(
        f"Epoch {epoch}/{config.epochs} | "
        f"train total {train_loss['total']:.4f} rec {train_loss['rec']:.4f} cls {train_loss['cls']:.4f} acc {train_loss['accuracy']:.4f} | "
        f"val total {val_loss['total']:.4f} rec {val_loss['rec']:.4f} cls {val_loss['cls']:.4f} acc {val_loss['accuracy']:.4f} | "
        f"{duration:.1f} min"
    )

    if val_loss['accuracy'] > best_acc:
        best_acc = val_loss['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_loss,
            'config': config,
            'lambda_cls': LAMBDA_CLS,
        }, best_path)
        print(f'Saved best to {best_path}')
